## Modelagem e Avaliação de Modelos

Nesta etapa, vamos **explorar, experimentar e entender** o processo de modelagem para previsão do preço das casas (dataset Ames Housing).

O objetivo aqui **não é** gerar o script final de produção, mas sim:
- Entender o impacto de cada escolha (scaler, modelo, hiperparâmetros)
- Comparar abordagens diferentes
- Visualizar os resultados de forma rica
- Documentar as decisões tomadas

O script final `train_model.py` será gerado com base nas conclusões deste notebook.

---
## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

RANDOM_STATE = 101

---
## Carregamento e Divisão dos Dados

In [ ]:
df = pd.read_csv('../data/processed/Ames_Final_DF.csv')

In [ ]:
df.head()

In [ ]:
X = df.drop('SalePrice', axis=1)
y = df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=RANDOM_STATE
)

print(f'Treino : {X_train.shape}')
print(f'Teste  : {X_test.shape}')

---
## Experimento 1 — Baseline: ElasticNet com GridSearchCV

Começamos com a abordagem direta: ElasticNet dentro de um `Pipeline` com `StandardScaler`.

**Por que Pipeline?**  
O scaler é re-ajustado em cada fold do cross-validation, evitando *data leakage* — o que é tecnicamente mais correto do que escalar antes do CV.

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', ElasticNet(max_iter=100000, random_state=RANDOM_STATE))
])

param_grid = {
    'model__alpha':    [.1, 1, 5, 10, 100],
    'model__l1_ratio': [.1, .2, .5, .8, .9, 1],
}

grid_model = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=5,
    verbose=0,
)

grid_model.fit(X_train, y_train)
print('Melhores parâmetros:', grid_model.best_params_)

y_pred = grid_model.predict(X_test)

print(f'MAE: {mean_absolute_error(y_test, y_pred):.2f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}')



### Visualizando os resultados do Grid Search

Vale olhar o heatmap de desempenho para cada combinação de `alpha` × `l1_ratio`.

In [ ]:
results = pd.DataFrame(grid_model.cv_results_)
results['RMSE'] = np.sqrt(-results['mean_test_score'])
results['alpha'] = results['param_model__alpha']
results['l1_ratio'] = results['param_model__l1_ratio']

pivot = results.pivot_table(index='alpha', columns='l1_ratio', values='RMSE')

plt.figure(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd_r', linewidths=0.5)
plt.title('RMSE para diferentes combinações de alpha e l1_ratio')
plt.tight_layout()
plt.show()


### Predito vs. Real

Um scatter plot mostra se o modelo erra mais em casas baratas ou caras.

In [ ]:
fig = plt.subplots(figsize=(14, 5))

# Predito vs Real
sns.scatterplot(x=y_test, y=y_pred, alpha=0.4, color='steelblue', edgecolors='white', s=30)
plt.xlabel('Preço real ($)')
plt.ylabel('Preço predito ($)')
plt.title('Predito vs. Real')


plt.tight_layout()
plt.show()

## Conclusões e Decisões para o `train_model.py`

| Decisão | Escolha | Motivo |
|---|---|---|
| Modelo | `LASSO` | O código comunica a intenção diretamente, O GridSearch fica mais rápido e limpo, remove um hiperparâmetro do grid (l1_ratio) |
| Scaler | `StandardScaler` dentro do Pipeline | Evita data leakage no CV |
| Hiperparâmetros | `alpha=100, l1_ratio=1` | Melhor resultado no GridSearchCV |